# Chapter 17: Nonlinear Kalman Filters

<a href="../lite/lab/index.html?path=ch17_nonlinear_kalman.ipynb" target="_blank" style="display:inline-block;padding:8px 18px;background:#1976d2;color:white;border-radius:5px;text-decoration:none;font-weight:bold;font-size:0.95em;">&#9654; Open in JupyterLite: run and edit this notebook</a>

*Runs entirely in your browser, no installation required.*

**How to use:** Edit the parameter values in each cell and re-run it to explore.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.linalg import cholesky, block_diag

%matplotlib inline
plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

The Kalman filter is beautiful but fragile. It assumes everything is linear. Real robots turn corners, sensors measure angles, and the world is curved. The Extended Kalman Filter patches this by pretending the world is locally flat. The Unscented Kalman Filter says: "Why linearize at all? Just pass a few carefully chosen points through the real function."

This chapter explores both approaches, compares their strengths and weaknesses, and gives you working implementations of each.

```{admonition} What you will build
:class: tip

- Implement the Extended Kalman Filter (EKF) with analytical Jacobians for a range-bearing sensor
- Implement the Unscented Kalman Filter (UKF) with sigma points
- Compare EKF and UKF on a robot tracking problem and measure which handles nonlinearity better
- Understand when linearization error makes the EKF unreliable

**Real world application:** The EKF is the workhorse of real time robotics (GPS/INS, visual odometry, robot localization). The UKF is its more robust cousin. After this chapter, you can implement both for any nonlinear system.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **FilterPy.kalman.ExtendedKalmanFilter** | Python EKF implementation |
| **FilterPy.kalman.UnscentedKalmanFilter** | Python UKF with configurable sigma point selection |
| **robot_localization (ROS 2)** | Supports both EKF and UKF modes for sensor fusion |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **FilterPy.kalman.ExtendedKalmanFilter** | Python EKF implementation |
| **FilterPy.kalman.UnscentedKalmanFilter** | Python UKF with configurable sigma point selection |
| **robot_localization (ROS 2)** | Supports both EKF and UKF modes for sensor fusion |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

## Opening Demo: Tracking a Robot on a Circle

A robot drives in a circle. Three landmarks sit at known positions. The robot measures **range** and **bearing** to each landmark. Both the motion model (constant turn rate) and the observation model (range and bearing) are **nonlinear**. We track the robot with both EKF and UKF and compare.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
dt_demo         = 0.1      # time step (s)                    (try 0.05, 0.1, 0.2)
v_demo          = 2.0      # forward speed (m/s)              (try 1.0, 2.0, 4.0)
omega_demo      = 0.3      # turn rate (rad/s)                (try 0.1, 0.3, 0.6)
n_steps_demo    = 200      # number of time steps             (try 100, 200, 400)
range_noise_std = 0.5      # range sensor noise (m)           (try 0.1, 0.5, 2.0)
bear_noise_std  = 0.1      # bearing sensor noise (rad)       (try 0.02, 0.1, 0.3)
proc_noise_v    = 0.3      # process noise on velocity        (try 0.05, 0.3, 1.0)
proc_noise_w    = 0.05     # process noise on turn rate       (try 0.01, 0.05, 0.2)
landmarks_demo  = np.array([[5, 10], [-5, 8], [0, -5]])  # landmark positions
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(42)

def motion_model(state, v, omega, dt):
    """Nonlinear motion: [x, y, theta] with constant velocity and turn rate."""
    x, y, theta = state
    x_new = x + v * np.cos(theta) * dt
    y_new = y + v * np.sin(theta) * dt
    theta_new = theta + omega * dt
    return np.array([x_new, y_new, theta_new])

def observation_model(state, landmark):
    """Range-bearing observation: returns [range, bearing]."""
    dx = landmark[0] - state[0]
    dy = landmark[1] - state[1]
    r = np.sqrt(dx**2 + dy**2)
    phi = np.arctan2(dy, dx) - state[2]
    phi = (phi + np.pi) % (2 * np.pi) - np.pi  # wrap to [-pi, pi]
    return np.array([r, phi])

def motion_jacobian(state, v, dt):
    """Jacobian of motion model w.r.t. state."""
    _, _, theta = state
    F = np.array([
        [1, 0, -v * np.sin(theta) * dt],
        [0, 1,  v * np.cos(theta) * dt],
        [0, 0,  1]
    ])
    return F

def observation_jacobian(state, landmark):
    """Jacobian of observation model w.r.t. state."""
    dx = landmark[0] - state[0]
    dy = landmark[1] - state[1]
    q = dx**2 + dy**2
    r = np.sqrt(q)
    H = np.array([
        [-dx / r,   -dy / r,    0],
        [ dy / q,   -dx / q,   -1]
    ])
    return H

# --- Generate ground truth ---
true_states = np.zeros((n_steps_demo + 1, 3))
true_states[0] = [0, 0, np.pi / 2]

for t in range(n_steps_demo):
    true_states[t + 1] = motion_model(true_states[t], v_demo, omega_demo, dt_demo)
    true_states[t + 1, :2] += np.random.normal(0, proc_noise_v * dt_demo, 2)
    true_states[t + 1, 2] += np.random.normal(0, proc_noise_w * dt_demo)

# --- Generate measurements ---
R_obs = np.diag([range_noise_std**2, bear_noise_std**2])
measurements = []
for t in range(1, n_steps_demo + 1):
    z_all = []
    for lm in landmarks_demo:
        z_true = observation_model(true_states[t], lm)
        z_noisy = z_true + np.random.multivariate_normal([0, 0], R_obs)
        z_all.append(z_noisy)
    measurements.append(z_all)

# --- EKF ---
Q_proc = np.diag([proc_noise_v**2 * dt_demo, proc_noise_v**2 * dt_demo, proc_noise_w**2 * dt_demo])
mu_ekf = np.array([0.5, 0.5, np.pi / 2 + 0.1])  # slightly wrong initial
Sigma_ekf = np.eye(3) * 1.0
ekf_estimates = [mu_ekf.copy()]

for t in range(n_steps_demo):
    # Predict
    F = motion_jacobian(mu_ekf, v_demo, dt_demo)
    mu_ekf = motion_model(mu_ekf, v_demo, omega_demo, dt_demo)
    Sigma_ekf = F @ Sigma_ekf @ F.T + Q_proc

    # Update with each landmark
    for k, lm in enumerate(landmarks_demo):
        z = measurements[t][k]
        z_pred = observation_model(mu_ekf, lm)
        H = observation_jacobian(mu_ekf, lm)
        S = H @ Sigma_ekf @ H.T + R_obs
        K = Sigma_ekf @ H.T @ np.linalg.inv(S)
        innov = z - z_pred
        innov[1] = (innov[1] + np.pi) % (2 * np.pi) - np.pi  # wrap bearing
        mu_ekf = mu_ekf + K @ innov
        Sigma_ekf = (np.eye(3) - K @ H) @ Sigma_ekf

    ekf_estimates.append(mu_ekf.copy())

ekf_estimates = np.array(ekf_estimates)

# --- UKF ---
def sigma_points(mu, Sigma, alpha=1e-3, beta=2.0, kappa=0.0):
    """Generate sigma points and weights for UKF."""
    n = len(mu)
    lam = alpha**2 * (n + kappa) - n
    sqrtP = cholesky((n + lam) * Sigma, lower=True)
    sigmas = np.zeros((2 * n + 1, n))
    sigmas[0] = mu
    for i in range(n):
        sigmas[i + 1]     = mu + sqrtP[:, i]
        sigmas[n + i + 1] = mu - sqrtP[:, i]
    # Weights
    Wm = np.full(2 * n + 1, 1.0 / (2 * (n + lam)))
    Wc = np.full(2 * n + 1, 1.0 / (2 * (n + lam)))
    Wm[0] = lam / (n + lam)
    Wc[0] = lam / (n + lam) + (1 - alpha**2 + beta)
    return sigmas, Wm, Wc

mu_ukf = np.array([0.5, 0.5, np.pi / 2 + 0.1])
Sigma_ukf = np.eye(3) * 1.0
ukf_estimates = [mu_ukf.copy()]

for t in range(n_steps_demo):
    # Predict
    sigs, Wm, Wc = sigma_points(mu_ukf, Sigma_ukf)
    sigs_pred = np.array([motion_model(s, v_demo, omega_demo, dt_demo) for s in sigs])
    mu_ukf = Wm @ sigs_pred
    Sigma_ukf = Q_proc.copy()
    for i in range(len(Wm)):
        d = sigs_pred[i] - mu_ukf
        Sigma_ukf += Wc[i] * np.outer(d, d)

    # Update with each landmark
    for k, lm in enumerate(landmarks_demo):
        z = measurements[t][k]
        sigs, Wm, Wc = sigma_points(mu_ukf, Sigma_ukf)
        z_sigs = np.array([observation_model(s, lm) for s in sigs])
        z_mean = Wm @ z_sigs
        S = R_obs.copy()
        Pxz = np.zeros((3, 2))
        for i in range(len(Wm)):
            dz = z_sigs[i] - z_mean
            dz[1] = (dz[1] + np.pi) % (2 * np.pi) - np.pi
            dx = sigs[i] - mu_ukf
            S += Wc[i] * np.outer(dz, dz)
            Pxz += Wc[i] * np.outer(dx, dz)
        K = Pxz @ np.linalg.inv(S)
        innov = z - z_mean
        innov[1] = (innov[1] + np.pi) % (2 * np.pi) - np.pi
        mu_ukf = mu_ukf + K @ innov
        Sigma_ukf = Sigma_ukf - K @ S @ K.T

    ukf_estimates.append(mu_ukf.copy())

ukf_estimates = np.array(ukf_estimates)

# --- Plot ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.plot(true_states[:, 0], true_states[:, 1], 'k-', lw=2, label='True path', zorder=5)
ax1.plot(ekf_estimates[:, 0], ekf_estimates[:, 1], 'steelblue', lw=1.5, label='EKF', zorder=4)
ax1.plot(ukf_estimates[:, 0], ukf_estimates[:, 1], 'tomato', lw=1.5, ls='--', label='UKF', zorder=4)
for lm in landmarks_demo:
    ax1.plot(lm[0], lm[1], 's', color='orange', markersize=12, markeredgecolor='k', zorder=6)
ax1.plot(true_states[0, 0], true_states[0, 1], 'go', markersize=10, label='Start', zorder=6)
ax1.set_xlabel('x (m)'); ax1.set_ylabel('y (m)')
ax1.set_title('Tracking a Robot on a Circular Path')
ax1.legend(fontsize=9); ax1.set_aspect('equal')

# Position error over time
ekf_err = np.sqrt(np.sum((ekf_estimates[:, :2] - true_states[:, :2])**2, axis=1))
ukf_err = np.sqrt(np.sum((ukf_estimates[:, :2] - true_states[:, :2])**2, axis=1))
steps = np.arange(n_steps_demo + 1)
ax2.plot(steps, ekf_err, 'steelblue', lw=1.5, label=f'EKF (RMSE={np.sqrt(np.mean(ekf_err**2)):.3f})')
ax2.plot(steps, ukf_err, 'tomato', lw=1.5, label=f'UKF (RMSE={np.sqrt(np.mean(ukf_err**2)):.3f})')
ax2.set_xlabel('Time step'); ax2.set_ylabel('Position error (m)')
ax2.set_title('Position Error Comparison')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 17.1 Nonlinearity: Why the Linear KF Fails

The standard Kalman filter assumes linear models:

$$\mathbf{x}_{t} = F\mathbf{x}_{t-1} + B\mathbf{u}_t + \mathbf{w}_t \qquad \mathbf{z}_t = H\mathbf{x}_t + \mathbf{v}_t$$

But most real systems are **nonlinear**:

$$\mathbf{x}_t = g(\mathbf{x}_{t-1}, \mathbf{u}_t) + \mathbf{w}_t \qquad \mathbf{z}_t = h(\mathbf{x}_t) + \mathbf{v}_t$$

Common nonlinearities in robotics:
- **Motion:** $x_{t+1} = x_t + v \cos\theta \cdot \Delta t$ (trigonometric)
- **Range:** $r = \sqrt{(x_l - x)^2 + (y_l - y)^2}$ (square root)
- **Bearing:** $\phi = \arctan2(y_l - y, x_l - x) - \theta$ (arctangent)

When a Gaussian passes through a nonlinear function, the result is **not Gaussian**. The linear KF assumes it stays Gaussian, which causes errors. The visualization below shows this effect.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
mu_input       = 1.0       # mean of input Gaussian          (try 0.5, 1.0, 2.0)
sigma_input    = 0.5       # std of input Gaussian            (try 0.1, 0.5, 1.0)
n_samples_nl   = 10000     # number of Monte Carlo samples    (try 1000, 10000)
nonlinear_func = 'cubic'   # 'cubic', 'sine', 'sqrt'         (try each)
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(7)

funcs = {
    'cubic': (lambda x: x**3 - 2*x, r'$f(x) = x^3 - 2x$'),
    'sine':  (lambda x: 3 * np.sin(x), r'$f(x) = 3\sin(x)$'),
    'sqrt':  (lambda x: np.sqrt(np.abs(x) + 0.1), r'$f(x) = \sqrt{|x| + 0.1}$'),
}

f, f_label = funcs[nonlinear_func]

# Sample from input Gaussian
samples_in = np.random.normal(mu_input, sigma_input, n_samples_nl)
samples_out = f(samples_in)

# Linear approximation (first-order Taylor at mean)
f_at_mu = f(mu_input)
# Numerical derivative
eps = 1e-5
f_prime = (f(mu_input + eps) - f(mu_input - eps)) / (2 * eps)
linear_mean = f_at_mu
linear_std = abs(f_prime) * sigma_input

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Input distribution
axes[0].hist(samples_in, bins=80, density=True, color='steelblue', alpha=0.6, label='Samples')
x_grid = np.linspace(mu_input - 4*sigma_input, mu_input + 4*sigma_input, 200)
axes[0].plot(x_grid, (1/(sigma_input*np.sqrt(2*np.pi))) * np.exp(-0.5*((x_grid-mu_input)/sigma_input)**2),
             'k-', lw=2, label='True Gaussian')
axes[0].set_title('Input distribution (Gaussian)')
axes[0].set_xlabel('x'); axes[0].legend(fontsize=8)

# Nonlinear function
x_plot = np.linspace(mu_input - 4*sigma_input, mu_input + 4*sigma_input, 300)
axes[1].plot(x_plot, f(x_plot), 'k-', lw=2, label=f_label)
axes[1].plot(x_plot, f_at_mu + f_prime * (x_plot - mu_input), 'tomato', lw=2, ls='--',
             label='Linear approx')
axes[1].axvline(mu_input, color='steelblue', ls=':', alpha=0.5)
axes[1].set_title(f'Function {f_label}')
axes[1].set_xlabel('x'); axes[1].set_ylabel('f(x)'); axes[1].legend(fontsize=8)

# Output distribution
axes[2].hist(samples_out, bins=80, density=True, color='forestgreen', alpha=0.6, label='True output')
y_grid = np.linspace(linear_mean - 4*linear_std, linear_mean + 4*linear_std, 200)
axes[2].plot(y_grid, (1/(linear_std*np.sqrt(2*np.pi)+1e-15)) * np.exp(-0.5*((y_grid-linear_mean)/(linear_std+1e-15))**2),
             'tomato', lw=2, ls='--', label='Linear approx (EKF)')
axes[2].axvline(np.mean(samples_out), color='forestgreen', ls='-', lw=2, alpha=0.7, label=f'True mean = {np.mean(samples_out):.2f}')
axes[2].axvline(linear_mean, color='tomato', ls='--', lw=2, alpha=0.7, label=f'EKF mean = {linear_mean:.2f}')
axes[2].set_title('Output distribution (no longer Gaussian!)')
axes[2].set_xlabel('f(x)'); axes[2].legend(fontsize=8)

plt.suptitle('Nonlinearity: Gaussian in, non-Gaussian out', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'True output mean:  {np.mean(samples_out):.4f}')
print(f'EKF output mean:   {linear_mean:.4f}  (error: {abs(np.mean(samples_out) - linear_mean):.4f})')
print(f'True output std:   {np.std(samples_out):.4f}')
print(f'EKF output std:    {linear_std:.4f}  (error: {abs(np.std(samples_out) - linear_std):.4f})')

## 17.2 Extended Kalman Filter (EKF)

The EKF handles nonlinearity by **linearizing** the motion and observation functions at the current estimate using Jacobians.

**Prediction:**
$$\bar{\boldsymbol{\mu}}_t = g(\boldsymbol{\mu}_{t-1}, \mathbf{u}_t) \qquad \bar{\boldsymbol{\Sigma}}_t = G_t \boldsymbol{\Sigma}_{t-1} G_t^\top + Q$$

where $G_t = \frac{\partial g}{\partial \mathbf{x}}\big|_{\boldsymbol{\mu}_{t-1}, \mathbf{u}_t}$ is the **Jacobian** of the motion model.

**Update:**
$$K_t = \bar{\boldsymbol{\Sigma}}_t H_t^\top (H_t \bar{\boldsymbol{\Sigma}}_t H_t^\top + R)^{-1}$$
$$\boldsymbol{\mu}_t = \bar{\boldsymbol{\mu}}_t + K_t(\mathbf{z}_t - h(\bar{\boldsymbol{\mu}}_t))$$
$$\boldsymbol{\Sigma}_t = (I - K_t H_t) \bar{\boldsymbol{\Sigma}}_t$$

where $H_t = \frac{\partial h}{\partial \mathbf{x}}\big|_{\bar{\boldsymbol{\mu}}_t}$ is the **Jacobian** of the observation model.

The key idea: use the actual nonlinear functions $g$ and $h$ to propagate the mean, but use their Jacobians to propagate the covariance.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
dt_ekf          = 0.1       # time step (s)                  (try 0.05, 0.1, 0.2)
v_ekf           = 1.5       # speed (m/s)                    (try 0.5, 1.5, 3.0)
omega_ekf       = 0.2       # turn rate (rad/s)              (try 0.1, 0.2, 0.5)
n_steps_ekf     = 150       # number of steps                (try 50, 150, 300)
Q_std_ekf       = [0.2, 0.2, 0.03]  # process noise [x, y, theta]
R_std_ekf       = [0.4, 0.08]        # measurement noise [range, bearing]
landmarks_ekf   = np.array([[6, 6], [-4, 8], [2, -6], [-6, -3]])  # 4 landmarks
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(99)

Q_ekf = np.diag(np.array(Q_std_ekf)**2)
R_ekf = np.diag(np.array(R_std_ekf)**2)

# Generate truth
true_ekf = np.zeros((n_steps_ekf + 1, 3))
true_ekf[0] = [0, 0, np.pi / 4]
for t in range(n_steps_ekf):
    true_ekf[t + 1] = motion_model(true_ekf[t], v_ekf, omega_ekf, dt_ekf)
    true_ekf[t + 1] += np.random.multivariate_normal([0, 0, 0], Q_ekf * dt_ekf)

# Generate measurements
meas_ekf = []
for t in range(1, n_steps_ekf + 1):
    z_list = []
    for lm in landmarks_ekf:
        z_true = observation_model(true_ekf[t], lm)
        z_noisy = z_true + np.random.multivariate_normal([0, 0], R_ekf)
        z_list.append(z_noisy)
    meas_ekf.append(z_list)

# Run EKF
mu = np.array([0.5, -0.5, np.pi / 4 + 0.2])
Sigma = np.eye(3) * 2.0
est_ekf = [mu.copy()]
cov_ekf = [Sigma.copy()]

for t in range(n_steps_ekf):
    # Predict
    G = motion_jacobian(mu, v_ekf, dt_ekf)
    mu = motion_model(mu, v_ekf, omega_ekf, dt_ekf)
    Sigma = G @ Sigma @ G.T + Q_ekf

    # Update
    for k, lm in enumerate(landmarks_ekf):
        z = meas_ekf[t][k]
        z_pred = observation_model(mu, lm)
        H = observation_jacobian(mu, lm)
        S = H @ Sigma @ H.T + R_ekf
        K = Sigma @ H.T @ np.linalg.inv(S)
        innov = z - z_pred
        innov[1] = (innov[1] + np.pi) % (2 * np.pi) - np.pi
        mu = mu + K @ innov
        Sigma = (np.eye(3) - K @ H) @ Sigma

    est_ekf.append(mu.copy())
    cov_ekf.append(Sigma.copy())

est_ekf = np.array(est_ekf)

def plot_cov_ellipse(ax, mu, cov, n_std=2, **kwargs):
    """Plot covariance ellipse."""
    vals, vecs = np.linalg.eigh(cov[:2, :2])
    vals = np.clip(vals, 1e-10, None)
    angle = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
    w, h = 2 * n_std * np.sqrt(vals[1]), 2 * n_std * np.sqrt(vals[0])
    ell = patches.Ellipse(mu[:2], w, h, angle=angle, **kwargs)
    ax.add_patch(ell)

fig, ax = plt.subplots(figsize=(9, 8))
ax.plot(true_ekf[:, 0], true_ekf[:, 1], 'k-', lw=2, label='True path', zorder=5)
ax.plot(est_ekf[:, 0], est_ekf[:, 1], 'steelblue', lw=1.5, label='EKF estimate', zorder=4)

# Plot covariance ellipses every 10 steps
for i in range(0, n_steps_ekf + 1, max(1, n_steps_ekf // 10)):
    plot_cov_ellipse(ax, est_ekf[i], cov_ekf[i], n_std=2,
                     facecolor='steelblue', edgecolor='steelblue', alpha=0.12)

for lm in landmarks_ekf:
    ax.plot(lm[0], lm[1], 's', color='orange', markersize=12, markeredgecolor='k', zorder=6)
ax.plot(true_ekf[0, 0], true_ekf[0, 1], 'go', markersize=10, label='Start', zorder=6)
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title('EKF Tracking with 2$\\sigma$ Covariance Ellipses')
ax.legend(fontsize=9); ax.set_aspect('equal')
plt.tight_layout()
plt.show()

rmse_ekf = np.sqrt(np.mean(np.sum((est_ekf[:, :2] - true_ekf[:, :2])**2, axis=1)))
print(f'EKF RMSE: {rmse_ekf:.4f} m')

## 17.3 Linearization Error

The EKF approximates the nonlinear function with its first order Taylor expansion. This works well when:
- The function is nearly linear over the range of the uncertainty
- The uncertainty (covariance) is small

It breaks down when:
- The function is highly curved
- The uncertainty is large (the Gaussian spans a region where the function curves significantly)

Let us visualize this directly. We pass a Gaussian through a nonlinear function and compare the EKF approximation (linearized) with the true transformed distribution.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
sigma_values  = [0.2, 0.6, 1.5]   # three levels of uncertainty to compare
mu_lin        = 1.5                # mean of input Gaussian         (try 0.5, 1.5, 3.0)
n_mc_samples  = 50000              # Monte Carlo samples            (try 10000, 50000)
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(42)

# Nonlinear function: polar to Cartesian
# (r, theta) -> (r*cos(theta), r*sin(theta))
# We fix r=5 and vary theta
r_fixed = 5.0
f_x = lambda theta: r_fixed * np.cos(theta)
f_y = lambda theta: r_fixed * np.sin(theta)

fig, axes = plt.subplots(1, len(sigma_values), figsize=(5 * len(sigma_values), 5))

for ax, sig in zip(axes, sigma_values):
    # Monte Carlo: true transformed distribution
    theta_samples = np.random.normal(mu_lin, sig, n_mc_samples)
    x_mc = f_x(theta_samples)
    y_mc = f_y(theta_samples)

    # EKF linearization
    x_lin = f_x(mu_lin)
    y_lin = f_y(mu_lin)
    # Jacobian at mu
    J = np.array([[-r_fixed * np.sin(mu_lin)],
                  [ r_fixed * np.cos(mu_lin)]])
    P_in = np.array([[sig**2]])
    P_out = J @ P_in @ J.T  # 2x2 covariance

    ax.scatter(x_mc[::5], y_mc[::5], s=1, alpha=0.15, color='forestgreen', label='True (MC)')
    ax.plot(x_lin, y_lin, 'ro', markersize=8, zorder=5)
    ax.plot(np.mean(x_mc), np.mean(y_mc), 'g^', markersize=8, zorder=5)

    # EKF ellipse
    vals, vecs = np.linalg.eigh(P_out)
    vals = np.clip(vals, 1e-10, None)
    angle = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
    ell = patches.Ellipse((x_lin, y_lin), 2*2*np.sqrt(vals[1]), 2*2*np.sqrt(vals[0]),
                          angle=angle, facecolor='tomato', edgecolor='tomato', alpha=0.25, label='EKF 2$\\sigma$')
    ax.add_patch(ell)

    # Arc showing the true geometry
    theta_arc = np.linspace(mu_lin - 3*sig, mu_lin + 3*sig, 200)
    ax.plot(f_x(theta_arc), f_y(theta_arc), 'k--', lw=1, alpha=0.5)

    ax.set_aspect('equal')
    ax.set_title(f'$\\sigma_\\theta$ = {sig:.1f} rad', fontsize=11)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.legend(fontsize=7, loc='lower left')

fig.suptitle('Linearization Error: polar to Cartesian\n'
             'Red dot = EKF mean, green triangle = true mean',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Left: small uncertainty, linearization is accurate.')
print('Right: large uncertainty, the EKF ellipse badly misrepresents the true banana-shaped distribution.')

## 17.4 Unscented Kalman Filter (UKF)

The UKF avoids Jacobians entirely. Instead, it uses the **unscented transform**: pick a small set of carefully chosen **sigma points**, pass each one through the actual nonlinear function, and then reconstruct a Gaussian from the results.

For state dimension $n$, the UKF uses $2n + 1$ sigma points:

$$\boldsymbol{\chi}_0 = \boldsymbol{\mu}$$
$$\boldsymbol{\chi}_i = \boldsymbol{\mu} + \left(\sqrt{(n + \lambda)\boldsymbol{\Sigma}}\right)_i \quad i = 1, \dots, n$$
$$\boldsymbol{\chi}_{i+n} = \boldsymbol{\mu} - \left(\sqrt{(n + \lambda)\boldsymbol{\Sigma}}\right)_i \quad i = 1, \dots, n$$

where $\lambda = \alpha^2(n + \kappa) - n$ controls the spread.

Each sigma point is transformed: $\boldsymbol{\gamma}_i = f(\boldsymbol{\chi}_i)$

The new mean and covariance are reconstructed as weighted sums:
$$\boldsymbol{\mu}' = \sum_i w_i^{(m)} \boldsymbol{\gamma}_i \qquad \boldsymbol{\Sigma}' = \sum_i w_i^{(c)} (\boldsymbol{\gamma}_i - \boldsymbol{\mu}')(\boldsymbol{\gamma}_i - \boldsymbol{\mu}')^\top$$

The beauty: **no Jacobians needed**, and the approximation is accurate to second order (vs. first order for EKF).

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
sigma_ut     = 0.8     # input uncertainty (rad)     (try 0.2, 0.5, 0.8, 1.2)
alpha_ut     = 1.0     # sigma point spread          (try 0.01, 0.5, 1.0)
beta_ut      = 2.0     # prior knowledge (2 = Gaussian)  (try 0, 2)
kappa_ut     = 0.0     # secondary scaling           (try 0, 1, 3)
n_mc_ut      = 50000   # Monte Carlo samples
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(42)

# 1D unscented transform on polar-to-Cartesian
mu_theta = 1.5
P_theta = np.array([[sigma_ut**2]])
n_dim = 1
lam = alpha_ut**2 * (n_dim + kappa_ut) - n_dim

# Sigma points (1D: 3 points)
sqrt_term = np.sqrt((n_dim + lam) * P_theta[0, 0])
chi = np.array([mu_theta, mu_theta + sqrt_term, mu_theta - sqrt_term])

# Weights
Wm = np.array([lam / (n_dim + lam), 1/(2*(n_dim + lam)), 1/(2*(n_dim + lam))])
Wc = np.array([lam / (n_dim + lam) + (1 - alpha_ut**2 + beta_ut),
               1/(2*(n_dim + lam)), 1/(2*(n_dim + lam))])

# Transform sigma points
gamma = np.array([[f_x(c), f_y(c)] for c in chi])

# Reconstruct
mu_ut = Wm @ gamma
P_ut = np.zeros((2, 2))
for i in range(3):
    d = gamma[i] - mu_ut
    P_ut += Wc[i] * np.outer(d, d)

# EKF for comparison
x_ekf_c = f_x(mu_theta)
y_ekf_c = f_y(mu_theta)
J_c = np.array([[-r_fixed * np.sin(mu_theta)],
                [ r_fixed * np.cos(mu_theta)]])
P_ekf_c = J_c @ P_theta @ J_c.T

# Monte Carlo truth
theta_mc = np.random.normal(mu_theta, sigma_ut, n_mc_ut)
x_mc_c = f_x(theta_mc)
y_mc_c = f_y(theta_mc)
mu_mc = np.array([np.mean(x_mc_c), np.mean(y_mc_c)])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax_idx, (title, mu_est, P_est, color, label) in enumerate([
    ('EKF (linearized)', np.array([x_ekf_c, y_ekf_c]), P_ekf_c, 'tomato', 'EKF 2$\\sigma$'),
    ('UKF (sigma points)', mu_ut, P_ut, 'steelblue', 'UKF 2$\\sigma$'),
]):
    ax = axes[ax_idx]
    ax.scatter(x_mc_c[::5], y_mc_c[::5], s=1, alpha=0.1, color='forestgreen')

    vals, vecs = np.linalg.eigh(P_est)
    vals = np.clip(vals, 1e-10, None)
    angle = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
    ell = patches.Ellipse(mu_est, 2*2*np.sqrt(vals[1]), 2*2*np.sqrt(vals[0]),
                          angle=angle, facecolor=color, edgecolor=color, alpha=0.25, label=label)
    ax.add_patch(ell)

    ax.plot(mu_est[0], mu_est[1], 'o', color=color, markersize=10, zorder=5, label=f'{title} mean')
    ax.plot(mu_mc[0], mu_mc[1], 'g^', markersize=10, zorder=5, label='True mean (MC)')

    if ax_idx == 1:
        # Show sigma points
        ax.plot(gamma[:, 0], gamma[:, 1], 'kx', markersize=12, markeredgewidth=2,
                zorder=6, label='Sigma points')

    theta_arc = np.linspace(mu_theta - 3*sigma_ut, mu_theta + 3*sigma_ut, 200)
    ax.plot(f_x(theta_arc), f_y(theta_arc), 'k--', lw=1, alpha=0.3)

    ax.set_aspect('equal')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.legend(fontsize=8)

fig.suptitle(f'EKF vs. UKF: polar to Cartesian ($\\sigma_\\theta$ = {sigma_ut} rad)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'True mean (MC):  ({mu_mc[0]:.3f}, {mu_mc[1]:.3f})')
print(f'EKF mean:        ({x_ekf_c:.3f}, {y_ekf_c:.3f})  error = {np.linalg.norm(np.array([x_ekf_c, y_ekf_c]) - mu_mc):.4f}')
print(f'UKF mean:        ({mu_ut[0]:.3f}, {mu_ut[1]:.3f})  error = {np.linalg.norm(mu_ut - mu_mc):.4f}')

## 17.5 Tradeoffs: EKF vs. UKF

| Property | EKF | UKF |
|---|---|---|
| **Jacobians required** | Yes | No |
| **Approximation order** | 1st order (Taylor) | 2nd order (unscented) |
| **Implementation complexity** | Must derive/code Jacobians | Need sigma point machinery |
| **Computational cost** | $O(n^2)$ per step | $O(n^3)$ (Cholesky) per step |
| **Numerical stability** | Can diverge if Jacobians are wrong | More robust |
| **Accuracy for mild nonlinearity** | Very good | Very good |
| **Accuracy for strong nonlinearity** | Degrades | Better than EKF |

**When to use EKF:** the system is mildly nonlinear, Jacobians are easy to derive, and computational cost matters (e.g., embedded systems).

**When to use UKF:** the system is significantly nonlinear, Jacobians are hard or impossible to derive, or you need better accuracy without much extra cost.

Let us compare both on the same tracking problem with varying levels of nonlinearity.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
omega_values    = [0.1, 0.3, 0.8]  # turn rates (more = more nonlinear)
v_compare       = 2.0              # forward speed (m/s)
dt_compare      = 0.1              # time step (s)
n_steps_compare = 150              # steps per run
n_trials        = 10               # Monte Carlo trials for averaging   (try 5, 10, 20)
lm_compare      = np.array([[6, 6], [-4, 8], [2, -6]])  # landmarks
Q_std_compare   = [0.2, 0.2, 0.03]
R_std_compare   = [0.5, 0.1]
# ─────────────────────────────────────────────────────────────────────────────

Q_cmp = np.diag(np.array(Q_std_compare)**2)
R_cmp = np.diag(np.array(R_std_compare)**2)

def run_ekf_trial(true_traj, meas_list, v, omega, dt, Q, R, lms):
    mu = np.array([0.5, 0.5, true_traj[0, 2] + 0.1])
    Sig = np.eye(3) * 1.0
    errs = []
    for t in range(len(meas_list)):
        G = motion_jacobian(mu, v, dt)
        mu = motion_model(mu, v, omega, dt)
        Sig = G @ Sig @ G.T + Q
        for k, lm in enumerate(lms):
            z = meas_list[t][k]
            z_pred = observation_model(mu, lm)
            H = observation_jacobian(mu, lm)
            S = H @ Sig @ H.T + R
            K = Sig @ H.T @ np.linalg.inv(S)
            innov = z - z_pred
            innov[1] = (innov[1] + np.pi) % (2*np.pi) - np.pi
            mu = mu + K @ innov
            Sig = (np.eye(3) - K @ H) @ Sig
        errs.append(np.sqrt(np.sum((mu[:2] - true_traj[t+1, :2])**2)))
    return np.array(errs)

def run_ukf_trial(true_traj, meas_list, v, omega, dt, Q, R, lms):
    mu = np.array([0.5, 0.5, true_traj[0, 2] + 0.1])
    Sig = np.eye(3) * 1.0
    errs = []
    for t in range(len(meas_list)):
        sigs_pts, Wm, Wc = sigma_points(mu, Sig)
        sigs_pred = np.array([motion_model(s, v, omega, dt) for s in sigs_pts])
        mu = Wm @ sigs_pred
        Sig = Q.copy()
        for i in range(len(Wm)):
            d = sigs_pred[i] - mu
            Sig += Wc[i] * np.outer(d, d)
        for k, lm in enumerate(lms):
            z = meas_list[t][k]
            sigs_pts, Wm, Wc = sigma_points(mu, Sig)
            z_sigs = np.array([observation_model(s, lm) for s in sigs_pts])
            z_mean = Wm @ z_sigs
            S = R.copy()
            Pxz = np.zeros((3, 2))
            for i in range(len(Wm)):
                dz = z_sigs[i] - z_mean
                dz[1] = (dz[1] + np.pi) % (2*np.pi) - np.pi
                dx = sigs_pts[i] - mu
                S += Wc[i] * np.outer(dz, dz)
                Pxz += Wc[i] * np.outer(dx, dz)
            K = Pxz @ np.linalg.inv(S)
            innov = z - z_mean
            innov[1] = (innov[1] + np.pi) % (2*np.pi) - np.pi
            mu = mu + K @ innov
            Sig = Sig - K @ S @ K.T
        errs.append(np.sqrt(np.sum((mu[:2] - true_traj[t+1, :2])**2)))
    return np.array(errs)

fig, axes = plt.subplots(1, len(omega_values), figsize=(5 * len(omega_values), 4))
rmse_results = []

for ax, omega_val in zip(axes, omega_values):
    ekf_rmses = []
    ukf_rmses = []
    for trial in range(n_trials):
        np.random.seed(trial * 100 + 7)
        # Generate truth
        tr = np.zeros((n_steps_compare + 1, 3))
        tr[0] = [0, 0, np.pi / 4]
        for t in range(n_steps_compare):
            tr[t+1] = motion_model(tr[t], v_compare, omega_val, dt_compare)
            tr[t+1] += np.random.multivariate_normal([0,0,0], Q_cmp * dt_compare)
        # Generate measurements
        ml = []
        for t in range(1, n_steps_compare + 1):
            zl = [observation_model(tr[t], lm) + np.random.multivariate_normal([0,0], R_cmp)
                  for lm in lm_compare]
            ml.append(zl)
        ekf_e = run_ekf_trial(tr, ml, v_compare, omega_val, dt_compare, Q_cmp, R_cmp, lm_compare)
        ukf_e = run_ukf_trial(tr, ml, v_compare, omega_val, dt_compare, Q_cmp, R_cmp, lm_compare)
        ekf_rmses.append(np.sqrt(np.mean(ekf_e**2)))
        ukf_rmses.append(np.sqrt(np.mean(ukf_e**2)))

    rmse_results.append((omega_val, np.mean(ekf_rmses), np.mean(ukf_rmses)))

    ax.bar(['EKF', 'UKF'], [np.mean(ekf_rmses), np.mean(ukf_rmses)],
           color=['steelblue', 'tomato'], edgecolor='white')
    ax.set_title(f'$\\omega$ = {omega_val} rad/s', fontsize=11)
    ax.set_ylabel('RMSE (m)')

fig.suptitle('EKF vs. UKF: RMSE at different turn rates\n(higher turn rate = stronger nonlinearity)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Turn rate | EKF RMSE | UKF RMSE | Winner')
for omega_val, ekf_r, ukf_r in rmse_results:
    winner = 'UKF' if ukf_r < ekf_r else 'EKF'
    print(f'  {omega_val:.1f}     |  {ekf_r:.4f}  |  {ukf_r:.4f}  |  {winner}')

## Capstone: Figure-8 Tracking with EKF and UKF

A robot drives a **figure-8** path, the most challenging trajectory because it involves constant direction changes with varying curvature. Four landmarks provide range-bearing measurements. We run both EKF and UKF, compare their RMSE, and plot covariance ellipses for both.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
dt_cap          = 0.05      # time step (s)                   (try 0.02, 0.05, 0.1)
n_steps_cap     = 500       # total steps                     (try 200, 500, 800)
fig8_scale      = 5.0       # size of figure-8                (try 3, 5, 8)
fig8_speed      = 0.02      # angular speed in parameter      (try 0.01, 0.02, 0.04)
Q_std_cap       = [0.15, 0.15, 0.02]  # process noise [x, y, theta]
R_std_cap       = [0.6, 0.12]          # measurement noise [range, bearing]
landmarks_cap   = np.array([[8, 8], [-8, 8], [8, -8], [-8, -8]])  # 4 landmarks
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(2024)

Q_cap = np.diag(np.array(Q_std_cap)**2)
R_cap = np.diag(np.array(R_std_cap)**2)

# Generate figure-8 trajectory
# Parametric: x(t) = scale * sin(t), y(t) = scale * sin(t) * cos(t)
true_cap = np.zeros((n_steps_cap + 1, 3))
param_t = np.linspace(0, 2 * np.pi * 2, n_steps_cap + 1)  # two full loops

for i in range(n_steps_cap + 1):
    t_p = param_t[i]
    true_cap[i, 0] = fig8_scale * np.sin(t_p)
    true_cap[i, 1] = fig8_scale * np.sin(t_p) * np.cos(t_p)
    # Heading from velocity
    dx = fig8_scale * np.cos(t_p)
    dy = fig8_scale * (np.cos(t_p)**2 - np.sin(t_p)**2)
    true_cap[i, 2] = np.arctan2(dy, dx)

# Add process noise
for i in range(1, n_steps_cap + 1):
    true_cap[i, :2] += np.random.multivariate_normal([0, 0], Q_cap[:2, :2] * dt_cap)
    true_cap[i, 2] += np.random.normal(0, Q_std_cap[2] * np.sqrt(dt_cap))

# Generate measurements
meas_cap = []
for t in range(1, n_steps_cap + 1):
    zl = [observation_model(true_cap[t], lm) + np.random.multivariate_normal([0,0], R_cap)
          for lm in landmarks_cap]
    meas_cap.append(zl)

# Run EKF
mu_e = true_cap[0].copy() + np.array([1.0, 1.0, 0.1])
Sig_e = np.eye(3) * 2.0
est_e_cap = [mu_e.copy()]
cov_e_cap = [Sig_e.copy()]

for t in range(n_steps_cap):
    # Use finite difference for velocity/omega from trajectory
    if t < n_steps_cap - 1:
        dx = true_cap[t+1, 0] - true_cap[t, 0]
        dy = true_cap[t+1, 1] - true_cap[t, 1]
        v_cmd = np.sqrt(dx**2 + dy**2) / dt_cap
        dtheta = true_cap[t+1, 2] - true_cap[t, 2]
        dtheta = (dtheta + np.pi) % (2*np.pi) - np.pi
        omega_cmd = dtheta / dt_cap
    # Predict
    G = motion_jacobian(mu_e, v_cmd, dt_cap)
    mu_e = motion_model(mu_e, v_cmd, omega_cmd, dt_cap)
    Sig_e = G @ Sig_e @ G.T + Q_cap
    # Update
    for k, lm in enumerate(landmarks_cap):
        z = meas_cap[t][k]
        z_pred = observation_model(mu_e, lm)
        H = observation_jacobian(mu_e, lm)
        S = H @ Sig_e @ H.T + R_cap
        K = Sig_e @ H.T @ np.linalg.inv(S)
        innov = z - z_pred
        innov[1] = (innov[1] + np.pi) % (2*np.pi) - np.pi
        mu_e = mu_e + K @ innov
        Sig_e = (np.eye(3) - K @ H) @ Sig_e
    est_e_cap.append(mu_e.copy())
    cov_e_cap.append(Sig_e.copy())

est_e_cap = np.array(est_e_cap)

# Run UKF
mu_u = true_cap[0].copy() + np.array([1.0, 1.0, 0.1])
Sig_u = np.eye(3) * 2.0
est_u_cap = [mu_u.copy()]
cov_u_cap = [Sig_u.copy()]

for t in range(n_steps_cap):
    if t < n_steps_cap - 1:
        dx = true_cap[t+1, 0] - true_cap[t, 0]
        dy = true_cap[t+1, 1] - true_cap[t, 1]
        v_cmd = np.sqrt(dx**2 + dy**2) / dt_cap
        dtheta = true_cap[t+1, 2] - true_cap[t, 2]
        dtheta = (dtheta + np.pi) % (2*np.pi) - np.pi
        omega_cmd = dtheta / dt_cap
    # Predict
    sigs_p, Wm_p, Wc_p = sigma_points(mu_u, Sig_u)
    sigs_pred_p = np.array([motion_model(s, v_cmd, omega_cmd, dt_cap) for s in sigs_p])
    mu_u = Wm_p @ sigs_pred_p
    Sig_u = Q_cap.copy()
    for i in range(len(Wm_p)):
        d = sigs_pred_p[i] - mu_u
        Sig_u += Wc_p[i] * np.outer(d, d)
    # Update
    for k, lm in enumerate(landmarks_cap):
        z = meas_cap[t][k]
        sigs_p, Wm_p, Wc_p = sigma_points(mu_u, Sig_u)
        z_sigs = np.array([observation_model(s, lm) for s in sigs_p])
        z_mean = Wm_p @ z_sigs
        S_u = R_cap.copy()
        Pxz_u = np.zeros((3, 2))
        for i in range(len(Wm_p)):
            dz = z_sigs[i] - z_mean
            dz[1] = (dz[1] + np.pi) % (2*np.pi) - np.pi
            dxs = sigs_p[i] - mu_u
            S_u += Wc_p[i] * np.outer(dz, dz)
            Pxz_u += Wc_p[i] * np.outer(dxs, dz)
        K_u = Pxz_u @ np.linalg.inv(S_u)
        innov = z - z_mean
        innov[1] = (innov[1] + np.pi) % (2*np.pi) - np.pi
        mu_u = mu_u + K_u @ innov
        Sig_u = Sig_u - K_u @ S_u @ K_u.T
    est_u_cap.append(mu_u.copy())
    cov_u_cap.append(Sig_u.copy())

est_u_cap = np.array(est_u_cap)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Trajectory comparison
ax = axes[0]
ax.plot(true_cap[:, 0], true_cap[:, 1], 'k-', lw=2, label='True path', zorder=5)
ax.plot(est_e_cap[:, 0], est_e_cap[:, 1], 'steelblue', lw=1.2, label='EKF', zorder=4)
ax.plot(est_u_cap[:, 0], est_u_cap[:, 1], 'tomato', lw=1.2, ls='--', label='UKF', zorder=4)
for lm in landmarks_cap:
    ax.plot(lm[0], lm[1], 's', color='orange', markersize=10, markeredgecolor='k', zorder=6)
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title('Figure-8 Trajectory Tracking'); ax.legend(fontsize=8); ax.set_aspect('equal')

# Covariance ellipses for both
ax = axes[1]
ax.plot(true_cap[:, 0], true_cap[:, 1], 'k-', lw=1.5, alpha=0.5, zorder=3)
step_interval = max(1, n_steps_cap // 15)
for i in range(0, n_steps_cap + 1, step_interval):
    plot_cov_ellipse(ax, est_e_cap[i], cov_e_cap[i], n_std=2,
                     facecolor='steelblue', edgecolor='steelblue', alpha=0.15)
    plot_cov_ellipse(ax, est_u_cap[i], cov_u_cap[i], n_std=2,
                     facecolor='tomato', edgecolor='tomato', alpha=0.15)
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title('Covariance Ellipses (blue=EKF, red=UKF)'); ax.set_aspect('equal')

# RMSE over time
ax = axes[2]
ekf_err_cap = np.sqrt(np.sum((est_e_cap[:, :2] - true_cap[:, :2])**2, axis=1))
ukf_err_cap = np.sqrt(np.sum((est_u_cap[:, :2] - true_cap[:, :2])**2, axis=1))
steps_cap = np.arange(n_steps_cap + 1)
ax.plot(steps_cap, ekf_err_cap, 'steelblue', lw=1, alpha=0.7,
        label=f'EKF (RMSE={np.sqrt(np.mean(ekf_err_cap**2)):.3f} m)')
ax.plot(steps_cap, ukf_err_cap, 'tomato', lw=1, alpha=0.7,
        label=f'UKF (RMSE={np.sqrt(np.mean(ukf_err_cap**2)):.3f} m)')
ax.set_xlabel('Time step'); ax.set_ylabel('Position error (m)')
ax.set_title('Position Error Over Time'); ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'EKF final RMSE: {np.sqrt(np.mean(ekf_err_cap**2)):.4f} m')
print(f'UKF final RMSE: {np.sqrt(np.mean(ukf_err_cap**2)):.4f} m')

## Exercises

**Exercise 17.1: Derive the Jacobians.**

For the motion model $g(x, y, \theta, v, \omega, \Delta t) = [x + v\cos\theta \cdot \Delta t,\; y + v\sin\theta \cdot \Delta t,\; \theta + \omega \cdot \Delta t]$, write out the $3 \times 3$ Jacobian $G = \partial g / \partial [x, y, \theta]$ by hand. Verify your answer by comparing with numerical finite differences. Do the same for the observation Jacobian $H$.

In [ ]:
# Your code here


**Exercise 17.2: Measurement frequency.**

Modify the EKF demo to only incorporate measurements every $k$ steps (skip the update step on other steps, only predict). Try $k = 1, 3, 5, 10$. Plot the RMSE over time for each. How does measurement frequency affect tracking quality? Is there a point where the EKF diverges?

In [ ]:
# Your code here


**Exercise 17.3: UKF sigma point parameters.**

Experiment with the UKF parameters $\alpha$, $\beta$, and $\kappa$. Run the figure-8 tracking with:
- $\alpha = 0.001, 0.1, 1.0$ (default $\beta=2, \kappa=0$)
- $\kappa = 0, 1, 3$ (default $\alpha=0.001, \beta=2$)

Record the RMSE for each combination. Which settings work best? Does $\alpha$ matter much for this problem?

In [ ]:
# Your code here


**Exercise 17.4: Landmark geometry.**

How does landmark placement affect filter performance? Test three configurations for the circular tracking problem:
1. All 4 landmarks clustered in one corner: $[(6,6), (7,7), (6,7), (7,6)]$
2. Spread around the path: $[(6,6), (-6,6), (6,-6), (-6,-6)]$
3. Only 2 landmarks: $[(6,6), (-6,-6)]$

Compare EKF RMSE for each. Explain why geometry matters.

In [ ]:
# Your code here


**Exercise 17.5 (Challenge): Implement the Iterated EKF (IEKF).**

The IEKF re-linearizes the observation model around the updated estimate and iterates the update step until convergence. Implement this:
1. After the standard EKF update, re-evaluate the Jacobian $H$ at the new estimate
2. Recompute $K$, re-apply the update
3. Repeat until $\|\Delta\mu\| < \epsilon$ (or a max of 5 iterations)

Compare IEKF vs. standard EKF on the figure-8 problem. Does iteration help? Under what conditions?

In [ ]:
# Your code here
